In [ ]:
!nvidia-smi

Fri Jul 24 19:28:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving crisis-aware-dialogue-cradle-response-final.zip to crisis-aware-dialogue-cradle-response-final.zip


In [ ]:
!mkdir -p /content/crisis-aware-dialogue
!unzip -q crisis-aware-dialogue-cradle-response-final.zip -d /content/crisis-aware-dialogue
%cd /content/crisis-aware-dialogue


/content/crisis-aware-dialogue


In [ ]:
!pwd
!ls
!find src -maxdepth 2 -type f | sort

/content/crisis-aware-dialogue
experiments  notebooks	README.md  reports  requirements.txt  src  tests
src/cradle_response/inference.py
src/cradle_response/__init__.py
src/cradle_response/preprocess.py
src/cradle_response/train_response_sft.py
src/download_data.py
src/__init__.py


In [1]:
from google.colab import files
uploaded = files.upload()

Saving crisis-aware-dialogue-cradle-response-final.zip to crisis-aware-dialogue-cradle-response-final.zip


In [2]:
!mkdir -p /content/crisis-aware-dialogue
!unzip -q crisis-aware-dialogue-cradle-response-final.zip -d /content/crisis-aware-dialogue
%cd /content/crisis-aware-dialogue

/content/crisis-aware-dialogue


In [3]:
!nvidia-smi

Fri Jul 24 20:22:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM GB: 39.5


In [5]:
%cd /content/crisis-aware-dialogue
!pwd


/content/crisis-aware-dialogue
/content/crisis-aware-dialogue


In [6]:
!pip install -q -r requirements.txt
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.9 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
Transformers: 5.13.1
Datasets: 5.0.0
PEFT: 0.19.1
TRL: 1.9.0
bitsandbytes: 0.49.2


In [8]:
!python -m unittest discover -s tests -v

test_builds_listener_targets_with_cumulative_risk (test_cradle_response_preprocess.CradleResponsePreprocessTests.test_builds_listener_targets_with_cumulative_risk) ... ok
test_classifier_interface_builds_generation_prompt (test_cradle_response_preprocess.CradleResponsePreprocessTests.test_classifier_interface_builds_generation_prompt) ... ok
test_redacts_common_contact_information (test_cradle_response_preprocess.CradleResponsePreprocessTests.test_redacts_common_contact_information) ... ok
test_short_history_still_starts_with_user (test_cradle_response_preprocess.CradleResponsePreprocessTests.test_short_history_still_starts_with_user) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.001s

OK


In [9]:
!python -m src.cradle_response.preprocess --max-history-turns 14

README.md: 100% 4.63k/4.63k [00:00<00:00, 10.2MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  67% 3.25M/4.84M [00:04<00:00, 2.59MB/s,  207kB/s  ]
data/train-00000-of-00001.parquet: downloading bytes: 100% 4.84M/4.84M [00:04<00:00, 1.04MB/s,  427kB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 4.84M/4.84M [00:04<00:00, 1.04MB/s,  435kB/s  ]

data/validation-00000-of-00001.parquet: downloading bytes:  69% 461k/669k [00:01<00:00, 233kB/s]
data/validation-00000-of-00001.parquet: downloading bytes: 100% 669k/669k [00:01<00:00, 337kB/s, 63.8kB/s  ]
data/validation-00000-of-00001.parquet: reconstructing file: 100% 669k/669k [00:01<00:00, 337kB/s, 63.8kB/s  ]

data/test-00000-of-00001.parquet: downloading bytes:  76% 794k/1.04M [00:03<00:00, 531kB/s, 34.3kB/s  ]
data/test-00000-of-00001.parquet: downloading bytes: 100% 1.04M/1.04M [00:03<00:00, 272kB/s, 93.6kB/s  ]
data/test-00000-of-00001.parquet: reconstructing file: 100% 1.04M/1.04M [00:03<00:00, 272kB/s, 9

In [10]:
from pathlib import Path

path = Path("src/cradle_response/preprocess.py")
source = path.read_text(encoding="utf-8")

helper = '''

def ensure_dialogue_ids(
    rows: Iterable[dict[str, Any]], *, split_name: str = "unknown"
) -> list[dict[str, Any]]:
    """Restore dialogue IDs when the Hub parquet omits its documented column."""
    materialized = [dict(row) for row in rows]

    if not materialized:
        return materialized

    has_ids = [
        "dialogue_id" in row and row["dialogue_id"] not in (None, "")
        for row in materialized
    ]

    if all(has_ids):
        return materialized

    if any(has_ids):
        raise ValueError("dialogue_id is present for only some rows")

    dialogue_index = -1

    for index, row in enumerate(materialized):
        if "turn_id" not in row:
            raise ValueError(f"Row {index} is missing required field: turn_id")

        turn_id = int(row["turn_id"])

        if turn_id == 0:
            dialogue_index += 1
        elif dialogue_index < 0:
            raise ValueError(
                "Cannot reconstruct dialogue IDs: first turn_id is not zero"
            )

        row["dialogue_id"] = f"{split_name}-{dialogue_index:06d}"

    return materialized
'''

if "def ensure_dialogue_ids(" not in source:
    anchor = "\ndef trim_history("
    source = source.replace(anchor, helper + anchor)

source = source.replace(
    'rows = [dict(row) for row in dataset[split_name]]',
    'rows = ensure_dialogue_ids(dataset[split_name], split_name=split_name)',
)

path.write_text(source, encoding="utf-8")
print("Patched:", path)


Patched: src/cradle_response/preprocess.py


In [11]:
!grep -n "ensure_dialogue_ids" src/cradle_response/preprocess.py

89:def ensure_dialogue_ids(
226:        rows = ensure_dialogue_ids(dataset[split_name], split_name=split_name)


In [12]:
!python -m src.cradle_response.preprocess --max-history-turns 14

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 269, in <module>
    main()
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 264, in main
    counts = preprocess_dataset(args.output_dir, max_history_turns=max_history_turns)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 228, in preprocess_dataset
    examples = build_response_examples(rows, max_history_turns=max_history_turns)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 169, in build_response_examples
    turn = parse_turn(row["text"])
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dial

In [13]:
from pathlib import Path

path = Path("src/cradle_response/preprocess.py")
source = path.read_text(encoding="utf-8")

source = source.replace(
    'def parse_turn(text: str) -> dict[str, str]:\n'
    '    """Convert a CRADLE text prefix into a chat-template role and content."""',
    'def parse_turn(text: str, *, role_hint: str | None = None) -> dict[str, str]:\n'
    '    """Convert a CRADLE turn to a chat role, using turn parity for bad rows."""',
)

old_block = '''    elif cleaned.startswith("Listener:"):
        role, content = "assistant", cleaned[len("Listener:") :].strip()
    else:
        raise ValueError(f"Turn does not start with User: or Listener:: {cleaned[:80]!r}")'''

new_block = '''    elif cleaned.startswith("Listener:"):
        role, content = "assistant", cleaned[len("Listener:") :].strip()
    elif role_hint in {"user", "assistant"}:
        role, content = role_hint, cleaned
    else:
        raise ValueError(f"Turn does not start with User: or Listener:: {cleaned[:80]!r}")'''

source = source.replace(old_block, new_block)

source = source.replace(
    '            turn = parse_turn(row["text"])',
    '            turn_id = int(row["turn_id"])\n'
    '            expected_role = "user" if turn_id % 2 == 0 else "assistant"\n'
    '            turn = parse_turn(row["text"], role_hint=expected_role)',
)

path.write_text(source, encoding="utf-8")
print("Role fallback patch applied")


Role fallback patch applied


In [14]:
!grep -n "role_hint\|expected_role" src/cradle_response/preprocess.py


50:def parse_turn(text: str, *, role_hint: str | None = None) -> dict[str, str]:
57:    elif role_hint in {"user", "assistant"}:
58:        role, content = role_hint, cleaned
172:            expected_role = "user" if turn_id % 2 == 0 else "assistant"
173:            turn = parse_turn(row["text"], role_hint=expected_role)


In [15]:
!python -m src.cradle_response.preprocess --max-history-turns 14


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 273, in <module>
    main()
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 268, in main
    counts = preprocess_dataset(args.output_dir, max_history_turns=max_history_turns)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 232, in preprocess_dataset
    examples = build_response_examples(rows, max_history_turns=max_history_turns)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 185, in build_response_examples
    raise ValueError(
ValueError: Listener turn 21 in dialogue train-002975 does not follow a User turn


In [16]:
from datasets import load_dataset

dataset = load_dataset("SungJoo/Cradle-Dialogue")
rows = dataset["train"]

dialogue_index = -1
target_rows = []

for row_index, row in enumerate(rows):
    if int(row["turn_id"]) == 0:
        dialogue_index += 1

    if dialogue_index == 2975:
        target_rows.append(
            {
                "row_index": row_index,
                "turn_id": row["turn_id"],
                "labels": row["labels"],
                "text": repr(row["text"][:300]),
            }
        )

    if dialogue_index > 2975:
        break

for row in target_rows:
    print(row)

{'row_index': 47215, 'turn_id': 0, 'labels': 'alert_past', 'text': "'User: Lately I keep having these “good” dreams about my ex—like he’s the perfect partner, kind, takes me out, great sex, the whole thing—and then I wake up confused and grossed out because real life was nothing like that. It’s been two years of no contact. Has anyone else had this? Any way to make t'"}
{'row_index': 47216, 'turn_id': 1, 'labels': '', 'text': "'Listener: That sounds really disorienting. I’m glad you posted. How often are these dreams happening, and how do they affect you during the day? If you’re comfortable sharing, what was the relationship actually like compared to the dreams?'"}
{'row_index': 47217, 'turn_id': 2, 'labels': 'confirm_DV_past', 'text': "'User: In real life it was the opposite. Our relationship was horribly violent—both physically and mentally. I filed a police report, reported him and got him expelled from our university, and I even planned to take him to court. It’s been two years wi

In [17]:
from pathlib import Path

path = Path("src/cradle_response/preprocess.py")
source = path.read_text(encoding="utf-8")

# Remove the previous parity fallback.
source = source.replace(
    'def parse_turn(text: str, *, role_hint: str | None = None) -> dict[str, str]:\n'
    '    """Convert a CRADLE turn to a chat role, using turn parity for bad rows."""',
    'def parse_turn(text: str) -> dict[str, str]:\n'
    '    """Convert a prefixed CRADLE turn into a chat-template message."""',
)

source = source.replace(
    '''    elif role_hint in {"user", "assistant"}:
        role, content = role_hint, cleaned
''',
    "",
)

source = source.replace(
    '''            turn_id = int(row["turn_id"])
            expected_role = "user" if turn_id % 2 == 0 else "assistant"
            turn = parse_turn(row["text"], role_hint=expected_role)''',
    '''            turn = parse_turn(row["text"])''',
)

# Add continuation-row merging.
helper = '''

def collapse_continuations(
    dialogue: list[dict[str, Any]]
) -> list[dict[str, Any]]:
    """Merge unprefixed bullet rows into the preceding message."""
    collapsed: list[dict[str, Any]] = []

    for row in dialogue:
        text = str(row["text"]).strip()

        if text.startswith(("User:", "Listener:")):
            collapsed.append(dict(row))
            continue

        if not collapsed:
            raise ValueError(
                "Dialogue begins with an unprefixed continuation row"
            )

        collapsed[-1]["text"] = (
            str(collapsed[-1]["text"]).rstrip() + "\\n" + text
        )

        extra_labels = parse_labels(row.get("labels"))

        if extra_labels:
            existing = parse_labels(collapsed[-1].get("labels"))
            collapsed[-1]["labels"] = "; ".join(
                dict.fromkeys(existing + extra_labels)
            )

    return collapsed
'''

if "def collapse_continuations(" not in source:
    source = source.replace(
        "\ndef parse_labels(",
        helper + "\ndef parse_labels(",
    )

source = source.replace(
    "    for dialogue in _ordered_dialogues(rows):",
    "    for raw_dialogue in _ordered_dialogues(rows):\n"
    "        dialogue = collapse_continuations(raw_dialogue)",
)

path.write_text(source, encoding="utf-8")
print("Continuation merge patch applied")

Continuation merge patch applied


In [19]:
!python -m py_compile src/cradle_response/preprocess.py

In [20]:
!python -m src.cradle_response.preprocess --max-history-turns 14

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 303, in <module>
    main()
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 298, in main
    counts = preprocess_dataset(args.output_dir, max_history_turns=max_history_turns)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 262, in preprocess_dataset
    examples = build_response_examples(rows, max_history_turns=max_history_turns)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/crisis-aware-dialogue/src/cradle_response/preprocess.py", line 215, in build_response_examples
    raise ValueError(
ValueError: Listener turn 0 in dialogue test-000003 does not follow a User turn


In [21]:
from datasets import load_dataset

dataset = load_dataset("SungJoo/Cradle-Dialogue")
rows = dataset["test"]

dialogue_index = -1
results = []

for row_index, row in enumerate(rows):
    if int(row["turn_id"]) == 0:
        dialogue_index += 1

    if 2 <= dialogue_index <= 4:
        results.append(
            {
                "dialogue_index": dialogue_index,
                "row_index": row_index,
                "turn_id": row["turn_id"],
                "labels": row["labels"],
                "text": repr(row["text"][:250]),
            }
        )

    if dialogue_index > 4:
        break

for row in results:
    print(row)

{'dialogue_index': 2, 'row_index': 28, 'turn_id': 0, 'labels': '', 'text': "'User: Hi… I’m not even sure if this is where I’m meant to be. I just… I’m so stuck.'"}
{'dialogue_index': 2, 'row_index': 29, 'turn_id': 1, 'labels': '', 'text': "'Listener: You’re in the right place. Take your time. What brought you in today?'"}
{'dialogue_index': 2, 'row_index': 30, 'turn_id': 2, 'labels': 'alert_ongoing', 'text': "'User: My partner and me, we keep having these huge arguments. Then the next morning he turns on the affection like nothing happened. It’s messing with my head.'"}
{'dialogue_index': 2, 'row_index': 31, 'turn_id': 3, 'labels': '', 'text': "'Listener: That sounds exhausting. How long has this been going on?'"}
{'dialogue_index': 2, 'row_index': 32, 'turn_id': 4, 'labels': '', 'text': "'User: Ages. It’s like… at night I’m bracing myself, and then in the morning he’s all hugs and “I love you” and I feel trapped all over again.'"}
{'dialogue_index': 2, 'row_index': 33, 'turn_id': 5, '

In [22]:
from pathlib import Path

path = Path("src/cradle_response/preprocess.py")
source = path.read_text(encoding="utf-8")

old = '''            if not history or history[-1]["role"] != "user":
                raise ValueError(
                    f"Listener turn {row['turn_id']} in dialogue {dialogue_id} "
                    "does not follow a User turn"
                )'''

new = '''            if not history:
                # Some test dialogues begin with a Listener greeting.
                # It has no preceding User input, so it is not a response target.
                continue

            if history[-1]["role"] != "user":
                raise ValueError(
                    f"Listener turn {row['turn_id']} in dialogue {dialogue_id} "
                    "does not follow a User turn"
                )'''

if old not in source:
    raise RuntimeError("Expected block was not found; stop before modifying")

source = source.replace(old, new)
path.write_text(source, encoding="utf-8")

print("Opening Listener handling applied")

Opening Listener handling applied


In [23]:
!python -m py_compile src/cradle_response/preprocess.py

In [24]:
!python -m src.cradle_response.preprocess --max-history-turns 14

Wrote CRADLE response SFT data to /content/crisis-aware-dialogue/data/processed/cradle_response: {'train': 22830, 'validation': 3142, 'test': 4327}


In [25]:
!ls -lh data/processed/cradle_response
!cat data/processed/cradle_response/manifest.json

total 73M
-rw-r--r-- 1 root root  667 Jul 24 20:38 manifest.json
-rw-r--r-- 1 root root  11M Jul 24 20:38 test.jsonl
-rw-r--r-- 1 root root  55M Jul 24 20:38 train.jsonl
-rw-r--r-- 1 root root 7.6M Jul 24 20:38 validation.jsonl
{
  "dataset": "SungJoo/Cradle-Dialogue",
  "task": "risk_conditioned_multi_turn_response_generation",
  "counts": {
    "train": 22830,
    "validation": 3142,
    "test": 4327
  },
  "dialogue_counts": {
    "train": 3058,
    "validation": 420,
    "test": 600
  },
  "max_history_turns": 14,
  "risk_interface": {
    "current_signals": "labels emitted for the latest User turn, or ['none']",
    "known_events": "unique crisis events detected earlier in this dialogue",
    "source": "gold_cradle_labels during training; classifier predictions in deployment"
  },
  "response_provenance": "GPT-5-generated Listener turns; not clinician-authored gold responses"
}

In [26]:
import json

with open(
    "data/processed/cradle_response/train.jsonl",
    encoding="utf-8",
) as f:
    example = json.loads(next(f))

print("Prompt roles:", [m["role"] for m in example["prompt"]])
print("Completion role:", example["completion"][0]["role"])
print("Dialogue ID:", example["metadata"]["dialogue_id"])
print("Target turn:", example["metadata"]["target_turn_id"])
print("Risk context:", example["metadata"]["risk_context"])
print("Prompt message count:", len(example["prompt"]))
print("Last user message:", example["prompt"][-1]["content"][:300])
print("Target response:", example["completion"][0]["content"][:300])

Prompt roles: ['system', 'user']
Completion role: assistant
Dialogue ID: train-000000
Target turn: 1
Risk context: {'current_signals': ['none'], 'known_events': [], 'source': 'gold_cradle_labels'}
Prompt message count: 2
Last user message: Hi—hope it's okay to post here. I'm looking to hear from anyone who's experienced sexual harassment from a colleague while working from home—like inappropriate texts, messages/comments during Zoom calls or Slack. Also from anyone who reported a colleague for sexual harassment before or during lockdo
Target response: Thanks for checking in. That sounds like an important piece. Are you a journalist or writing independently, and how are you planning to protect anonymity for folks who share?


In [27]:
import json

with open(
    "data/processed/cradle_response/train.jsonl",
    encoding="utf-8",
) as f:
    for line in f:
        item = json.loads(line)
        risk = item["metadata"]["risk_context"]

        if len(item["prompt"]) >= 6 and risk["known_events"]:
            print("Prompt roles:", [m["role"] for m in item["prompt"]])
            print("Target turn:", item["metadata"]["target_turn_id"])
            print("Risk context:", risk)
            print("Last user:", item["prompt"][-1]["content"][:300])
            print("Target:", item["completion"][0]["content"][:300])
            break

Prompt roles: ['system', 'user', 'assistant', 'user', 'assistant', 'user']
Target turn: 5
Risk context: {'current_signals': ['none'], 'known_events': ['alert_ongoing'], 'source': 'gold_cradle_labels'}
Last user: He still brings it up when we’re alone. He’s got a girlfriend and 5 kids, and the day I met her he asked me if I thought he was the hottest in the couple. I told him no. He just laughs it off like it’s funny.
Target: Ugh. Has he made comments about your body or tattoos, or said sexual things that felt hinting?


In [28]:
from huggingface_hub import login
login()


In [29]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    filename="config.json",
)

print("Llama access successful")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

Llama access successful


In [30]:
!python -m src.cradle_response.train_response_sft \
  --smoke-test \
  --output-dir outputs/cradle-response-smoke

Generating train split: 22830 examples [00:00, 64077.55 examples/s]
Generating validation split: 3142 examples [00:00, 71562.41 examples/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 14.9MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 51.0MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.20MB/s]

model.safetensors: downloading bytes:  13% 326M/2.47G [00:02<00:06, 338MB/s, 25.8MB/s  ]
model.safetensors: downloading bytes:  16% 387M/2.47G [00:02<00:05, 394MB/s, 30.4MB/s  ]
model.safetensors: downloading bytes:  18% 439M/2.47G [00:02<00:04, 421MB/s, 36.0MB/s  ]
model.safetensors: downloading bytes:  23% 564M/2.47G [00:02<00:03, 515MB/s, 45.3MB/s  ]
model.safetensors: downloading bytes:  27% 661M/2.47G [00:02<00:03, 501MB/s, 55.9MB/s  ]
model.safetensors: downloading bytes:  34% 845M/2.47G [00:02<00:02, 672MB/s, 68.2MB/s  ]
model.safetensors: downloading bytes:  39% 961M/2.47G [00:03<00:02, 632MB/s, 80.0MB/s  ]
model.safetensors: downloading bytes:  51% 1.26G/2.47

In [31]:
!python -m src.cradle_response.train_response_sft \
  --epochs 1 \
  --batch-size 4 \
  --gradient-accumulation-steps 4 \
  --max-length 2048 \
  --max-eval-samples 500 \
  --output-dir outputs/llama-3.2-1b-cradle-response-qlora-full

Loading weights:   1% 1/146 [00:00<00:19,  7.51it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 146/146 [00:00<00:00, 188.11it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/content/crisis-aware-dialogue/src/cradle_response/train_response_sft.py:90: FutureWarning: The `'keep_end'` truncation mode is deprecated and will be removed in v2.0.0. Use `truncation_mode='keep_start'` (the default) instead.
  training_args = SFTConfig(
Training on CRADLE Listener turns. These are synthetic GPT-5 responses, not clinician-authored ground truth.
Tokenizing train dataset: 100% 22830/22830 [01:13<00:00, 309.41 examples/s]
Building labels for train dataset: 100% 22830/22830 [00:15<00:00, 1478.62 examples/s]
Truncating tr

In [32]:
!ls -lh outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter
!du -sh outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter

total 60M
-rw-r--r-- 1 root root 1.1K Jul 24 21:28 adapter_config.json
-rw------- 1 root root  44M Jul 24 21:28 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K Jul 24 21:28 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Jul 24 21:28 README.md
-rw-r--r-- 1 root root  354 Jul 24 21:28 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jul 24 21:28 tokenizer.json
-rw-r--r-- 1 root root 5.6K Jul 24 21:28 training_args.bin
60M	outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter


In [33]:
import json
from pathlib import Path

history = [
    {
        "role": "user",
        "content": (
            "My partner has been getting increasingly angry during arguments, "
            "and I never know what mood he will be in."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "That sounds frightening and unpredictable. "
            "Has he threatened you or become physically aggressive?"
        ),
    },
]

Path("history.json").write_text(
    json.dumps(history, indent=2),
    encoding="utf-8",
)

print(Path("history.json").read_text())

[
  {
    "role": "user",
    "content": "My partner has been getting increasingly angry during arguments, and I never know what mood he will be in."
  },
  {
    "role": "assistant",
    "content": "That sounds frightening and unpredictable. Has he threatened you or become physically aggressive?"
  }
]


In [34]:
!python -m src.cradle_response.inference \
  --adapter outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter \
  --history-file history.json \
  --text "Last night he threw things across the room and blocked the door when I tried to leave. I am alone right now, but he may come back soon." \
  --current-signal confirm_DV_ongoing \
  --known-event alert_ongoing \
  --known-event confirm_DV_ongoing \
  --max-new-tokens 180

Loading weights: 100% 146/146 [00:00<00:00, 184.79it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Thank you for telling me. Are you safe right now, and do you have any injuries that need medical attention?


In [35]:
import json
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from src.cradle_response.inference import (
    build_messages,
    build_risk_context,
    load_history,
)

model_id = "meta-llama/Llama-3.2-1B-Instruct"

context = build_risk_context(
    current_signals=["confirm_DV_ongoing"],
    known_events=["alert_ongoing", "confirm_DV_ongoing"],
)

messages = build_messages(
    load_history(Path("history.json")),
    (
        "Last night he threw things across the room and blocked the door "
        "when I tried to leave. I am alone right now, but he may come back soon."
    ),
    context,
    14,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map="auto",
)

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(base_model.device)

with torch.inference_mode():
    output = base_model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output[0, inputs["input_ids"].shape[1]:]
base_response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True,
).strip()

print("BASE LLAMA RESPONSE:")
print(base_response)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


BASE LLAMA RESPONSE:
I'm so sorry to hear that you're in this situation. That sounds like a really scary and isolating experience for you. Have you considered talking to a trusted friend, family member, or authority figure about what's been happening? Someone who can offer you support and help you feel safe.


In [36]:
import json
import math
import torch

from pathlib import Path
from datasets import load_dataset
from peft import PeftModel
from tqdm.auto import tqdm


TEST_FILE = "data/processed/cradle_response/test.jsonl"
ADAPTER_PATH = (
    "outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter"
)
MAX_LENGTH = 2048
BATCH_SIZE = 4
NUM_TEST_EXAMPLES = 500


test_data = load_dataset(
    "json",
    data_files={"test": TEST_FILE},
)["test"]

# Fixed random subset for reproducibility.
test_subset = (
    test_data
    .shuffle(seed=42)
    .select(range(min(NUM_TEST_EXAMPLES, len(test_data))))
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def encode_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=True,
        add_generation_prompt=True,
    )

    full_messages = example["prompt"] + example["completion"]

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokens are not a prefix of full example")

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    # Same keep-end policy used during training.
    full_ids = full_ids[-MAX_LENGTH:]
    labels = labels[-MAX_LENGTH:]

    return {
        "input_ids": full_ids,
        "labels": labels,
    }


encoded_test = [encode_example(row) for row in test_subset]


def evaluate_completion(model, encoded_rows, name):
    model.eval()

    total_nll = 0.0
    total_tokens = 0
    correct_tokens = 0

    for start in tqdm(
        range(0, len(encoded_rows), BATCH_SIZE),
        desc=name,
    ):
        batch = encoded_rows[start:start + BATCH_SIZE]
        max_len = max(len(row["input_ids"]) for row in batch)

        input_ids = []
        labels = []
        attention_mask = []

        for row in batch:
            pad_length = max_len - len(row["input_ids"])

            input_ids.append(
                row["input_ids"]
                + [tokenizer.pad_token_id] * pad_length
            )
            labels.append(
                row["labels"] + [-100] * pad_length
            )
            attention_mask.append(
                [1] * len(row["input_ids"]) + [0] * pad_length
            )

        input_ids = torch.tensor(
            input_ids,
            dtype=torch.long,
            device=model.device,
        )
        labels = torch.tensor(
            labels,
            dtype=torch.long,
            device=model.device,
        )
        attention_mask = torch.tensor(
            attention_mask,
            dtype=torch.long,
            device=model.device,
        )

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

        shifted_labels = labels[:, 1:]
        valid_mask = shifted_labels.ne(-100)
        valid_tokens = int(valid_mask.sum().item())

        predictions = outputs.logits[:, :-1].argmax(dim=-1)
        correct_tokens += int(
            ((predictions == shifted_labels) & valid_mask).sum().item()
        )

        total_nll += float(outputs.loss.item()) * valid_tokens
        total_tokens += valid_tokens

        del input_ids, labels, attention_mask, outputs, predictions

    mean_loss = total_nll / total_tokens

    return {
        "model": name,
        "examples": len(encoded_rows),
        "completion_tokens": total_tokens,
        "loss": mean_loss,
        "perplexity": math.exp(mean_loss),
        "token_accuracy": correct_tokens / total_tokens,
    }


# Evaluate original base Llama first.
base_metrics = evaluate_completion(
    base_model,
    encoded_test,
    "base_llama",
)

print("BASE METRICS")
print(json.dumps(base_metrics, indent=2))


# Load the trained adapter onto the same frozen base model.
tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

tuned_metrics = evaluate_completion(
    tuned_model,
    encoded_test,
    "fine_tuned_llama",
)

print("FINE-TUNED METRICS")
print(json.dumps(tuned_metrics, indent=2))


comparison = {
    "dataset": "CRADLE-Dialogue held-out test",
    "selection": "500 examples shuffled with seed 42",
    "base": base_metrics,
    "fine_tuned": tuned_metrics,
    "loss_reduction_percent": (
        (base_metrics["loss"] - tuned_metrics["loss"])
        / base_metrics["loss"]
        * 100
    ),
}

metrics_path = Path(
    "outputs/llama-3.2-1b-cradle-response-qlora-full/"
    "test_comparison_metrics.json"
)

metrics_path.write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8",
)

print("COMPARISON")
print(json.dumps(comparison, indent=2))
print("Saved to:", metrics_path)

Generating test split: 0 examples [00:00, ? examples/s]

ValueError: Prompt tokens are not a prefix of full example

In [41]:
for example_index, example in enumerate(test_subset):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=True,
        add_generation_prompt=True,
    )

    full_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"],
        tokenize=True,
        add_generation_prompt=False,
    )

    common_length = 0

    for left, right in zip(prompt_ids, full_ids):
        if left != right:
            break
        common_length += 1

    if common_length != len(prompt_ids):
        print("Example index:", example_index)
        print("Prompt length:", len(prompt_ids))
        print("Full length:", len(full_ids))
        print("Common prefix length:", common_length)

        start = max(0, common_length - 20)
        end_prompt = min(len(prompt_ids), common_length + 20)
        end_full = min(len(full_ids), common_length + 20)

        print("\nPROMPT TEMPLATE AROUND MISMATCH:")
        print(tokenizer.decode(prompt_ids[start:end_prompt]))

        print("\nFULL TEMPLATE AROUND MISMATCH:")
        print(tokenizer.decode(full_ids[start:end_full]))

        print("\nTarget response:")
        print(example["completion"][0]["content"][:300])
        break

In [42]:
common_length = 0

for prompt_token, full_token in zip(prompt_ids, full_ids):
    if prompt_token != full_token:
        break
    common_length += 1

# A small difference at the assistant-template boundary is acceptable,
# but a large mismatch indicates a real formatting problem.
if common_length < len(prompt_ids) - 8:
    raise ValueError(
        f"Large chat-template mismatch: "
        f"prompt={len(prompt_ids)}, common={common_length}"
    )

labels = (
    [-100] * common_length
    + full_ids[common_length:]
)

In [46]:
def encode_example_tensor(example):
    prompt_encoding = tokenizer.apply_chat_template(
        example["prompt"],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    full_encoding = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"],
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt",
        return_dict=True,
    )

    prompt_ids = (
        prompt_encoding["input_ids"][0]
        .detach()
        .cpu()
        .tolist()
    )

    full_ids = (
        full_encoding["input_ids"][0]
        .detach()
        .cpu()
        .tolist()
    )

    common_length = 0

    for prompt_token, full_token in zip(prompt_ids, full_ids):
        if prompt_token != full_token:
            break
        common_length += 1

    if common_length < len(prompt_ids) - 8:
        raise ValueError(
            f"Large template mismatch: "
            f"prompt={len(prompt_ids)}, common={common_length}"
        )

    labels = (
        [-100] * common_length
        + full_ids[common_length:]
    )

    return {
        "input_ids": full_ids[-MAX_LENGTH:],
        "labels": labels[-MAX_LENGTH:],
    }


encoded_test = [
    encode_example_tensor(row)
    for row in test_subset
]

print("Re-encoded examples:", len(encoded_test))
print("First input type:", type(encoded_test[0]["input_ids"]))
print("First token type:", type(encoded_test[0]["input_ids"][0]))


base_metrics = evaluate_model(
    base_model,
    encoded_test,
    "base_llama",
)

print("\nBASE METRICS")
print(json.dumps(base_metrics, indent=2))


tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

tuned_metrics = evaluate_model(
    tuned_model,
    encoded_test,
    "fine_tuned_llama",
)

print("\nFINE-TUNED METRICS")
print(json.dumps(tuned_metrics, indent=2))


comparison = {
    "dataset": "CRADLE-Dialogue held-out test",
    "selection": "500 examples shuffled with seed 42",
    "base": base_metrics,
    "fine_tuned": tuned_metrics,
    "loss_reduction_percent": (
        (base_metrics["loss"] - tuned_metrics["loss"])
        / base_metrics["loss"]
        * 100
    ),
    "token_accuracy_improvement": (
        tuned_metrics["token_accuracy"]
        - base_metrics["token_accuracy"]
    ),
}

metrics_path = Path(
    "outputs/llama-3.2-1b-cradle-response-qlora-full/"
    "test_comparison_metrics.json"
)

metrics_path.write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8",
)

print("\nCOMPARISON")
print(json.dumps(comparison, indent=2))
print("\nSaved to:", metrics_path)

Re-encoded examples: 500
First input type: <class 'list'>
First token type: <class 'int'>


base_llama:   0%|          | 0/125 [00:00<?, ?it/s]


BASE METRICS
{
  "model": "base_llama",
  "examples": 500,
  "completion_tokens": 23951,
  "loss": 3.0833921722069517,
  "perplexity": 21.83233597088665,
  "token_accuracy": 0.3837418061876331
}


fine_tuned_llama:   0%|          | 0/125 [00:00<?, ?it/s]


FINE-TUNED METRICS
{
  "model": "fine_tuned_llama",
  "examples": 500,
  "completion_tokens": 23951,
  "loss": 2.032796191853634,
  "perplexity": 7.6354065988787445,
  "token_accuracy": 0.5118784184376435
}

COMPARISON
{
  "dataset": "CRADLE-Dialogue held-out test",
  "selection": "500 examples shuffled with seed 42",
  "base": {
    "model": "base_llama",
    "examples": 500,
    "completion_tokens": 23951,
    "loss": 3.0833921722069517,
    "perplexity": 21.83233597088665,
    "token_accuracy": 0.3837418061876331
  },
  "fine_tuned": {
    "model": "fine_tuned_llama",
    "examples": 500,
    "completion_tokens": 23951,
    "loss": 2.032796191853634,
    "perplexity": 7.6354065988787445,
    "token_accuracy": 0.5118784184376435
  },
  "loss_reduction_percent": 34.07273294079062,
  "token_accuracy_improvement": 0.12813661225001038
}

Saved to: outputs/llama-3.2-1b-cradle-response-qlora-full/test_comparison_metrics.json


In [47]:
!du -sh outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter
!ls -lh outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter

60M	outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter
total 60M
-rw-r--r-- 1 root root 1.1K Jul 24 21:28 adapter_config.json
-rw------- 1 root root  44M Jul 24 21:28 adapter_model.safetensors
-rw-r--r-- 1 root root 3.8K Jul 24 21:28 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Jul 24 21:28 README.md
-rw-r--r-- 1 root root  354 Jul 24 21:28 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jul 24 21:28 tokenizer.json
-rw-r--r-- 1 root root 5.6K Jul 24 21:28 training_args.bin


In [48]:
!zip -qr cradle-response-a100-final.zip \
  outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter \
  outputs/llama-3.2-1b-cradle-response-qlora-full/test_comparison_metrics.json

In [49]:
!ls -lh cradle-response-a100-final.zip
!unzip -l cradle-response-a100-final.zip | tail -n 20


-rw-r--r-- 1 root root 23M Jul 24 21:50 cradle-response-a100-final.zip
Archive:  cradle-response-a100-final.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/
 45118424  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/adapter_model.safetensors
      354  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/tokenizer_config.json
     3827  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/chat_template.jinja
     1111  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/adapter_config.json
     5230  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/README.md
 17209920  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-response-qlora-full/final_adapter/tokenizer.json
     5713  2026-07-24 21:28   outputs/llama-3.2-1b-cradle-re

In [50]:
from google.colab import files

files.download("cradle-response-a100-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>